> ## ⚠️  Superseded — single-run draft, retained as a record
>
> **This notebook will not run.** It opens `processed_data/all_data.pkl`, the earlier
> 53-feature preprocessing artefact that was replaced by `canonical_data.pkl` when the
> feature set was corrected. Nothing in this folder regenerates it, so the first cell raises
> `FileNotFoundError`.
>
> **No result reported in the thesis comes from this notebook.** Every reported figure is
> produced by the canonical path `00`/`01` → `08`–`15`, which trains over thirty fixed seeds
> (`SEEDS = [42 + 7*i for i in range(30)]`). Model definitions — `TemporalAttention`,
> `build_lstm_transformer`, `build_standalone_lstm` — live in `canonical_pipeline.py`.
>
> It is kept because its saved outputs are the record of the single-run baseline stage the
> project passed through. See README §5.

# 04 — Train Vanilla LSTM
Ablation baseline — LSTM only, no attention mechanism.

In [1]:
import pickle, numpy as np, warnings
warnings.filterwarnings('ignore')
import tensorflow as tf
from tensorflow.keras import layers, Model, Input
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
np.random.seed(42); tf.random.set_seed(42)

with open('processed_data/all_data.pkl', 'rb') as f:
    data = pickle.load(f)
X_tr_seq_, X_tr_feat_, y_tr_seq_ = data['X_tr_seq_'], data['X_tr_feat_'], data['y_tr_seq_']
X_val_seq, X_val_feat, y_val_seq = data['X_val_seq'], data['X_val_feat'], data['y_val_seq']
X_te_seq, X_te_feat = data['X_te_seq'], data['X_te_feat']
y_te_seq_raw = data['y_te_seq_raw']
target_scaler = data['target_scaler']
SEQ_LEN = data['SEQ_LEN']
n_features = X_tr_seq_.shape[2]
print(f"Train: {len(X_tr_seq_)}, Val: {len(X_val_seq)}, Test: {len(X_te_seq)}")
print(f"Shape: ({SEQ_LEN}, {n_features})")

Train: 2431, Val: 429, Test: 695
Shape: (28, 53)


In [2]:
# Build Vanilla LSTM
si = Input(shape=(SEQ_LEN, n_features), name='sequence_input')
fi = Input(shape=(n_features,), name='feature_input')
x = layers.LSTM(16, return_sequences=False, dropout=0.2)(si)
f = layers.Dense(16, activation='relu')(fi)
f = layers.Dropout(0.2)(f)
c = layers.Concatenate()([x, f])
h = layers.Dense(16, activation='relu')(c)
h = layers.Dropout(0.2)(h)
o = layers.Dense(1, activation='sigmoid')(h)
model = Model(inputs=[si, fi], outputs=o, name='Vanilla_LSTM')
model.compile(optimizer=Adam(learning_rate=0.001, clipnorm=1.0), loss='mse', metrics=['mae'])
print(f"Vanilla LSTM: {model.count_params():,} params")

Vanilla LSTM: 5,889 params


In [3]:
# Train
print("Training...")
_ = model.predict([X_tr_seq_[:1], X_tr_feat_[:1]], verbose=0)  # warmup

history = model.fit(
    [X_tr_seq_, X_tr_feat_], y_tr_seq_,
    validation_data=([X_val_seq, X_val_feat], y_val_seq),
    epochs=100, batch_size=32, verbose=0,
    callbacks=[
        EarlyStopping(monitor='val_loss', patience=20, restore_best_weights=True, verbose=1),
        ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=7, min_lr=1e-7, verbose=0),
    ])

pred_s = model.predict([X_te_seq, X_te_feat], verbose=0).flatten()
pred = np.maximum(target_scaler.inverse_transform(pred_s.reshape(-1,1)).flatten(), 0)
rmse = np.sqrt(mean_squared_error(y_te_seq_raw, pred))
mae = mean_absolute_error(y_te_seq_raw, pred)
r2 = r2_score(y_te_seq_raw, pred)

best = np.argmin(history.history['val_loss']) + 1
print(f"\n✅ Vanilla LSTM — Best epoch: {best}/{len(history.history['val_loss'])}")
print(f"   RMSE: {rmse:.2f}, MAE: {mae:.2f}, R²: {r2:.4f}")

Training...
Epoch 51: early stopping
Restoring model weights from the end of the best epoch: 31.

✅ Vanilla LSTM — Best epoch: 31/51
   RMSE: 27.41, MAE: 14.01, R²: 0.7651


In [4]:
results = {'model': 'Vanilla LSTM', 'RMSE': rmse, 'MAE': mae, 'R2': r2,
           'predictions': pred, 'history': history.history}
with open('metrics/vanilla_lstm_results.pkl', 'wb') as f:
    pickle.dump(results, f)
model.save('models/vanilla_lstm.keras')
print("✅ Saved")

✅ Saved
